# 03 - Audio Preprocessing & Feature Extraction

**Project:** Hassaniya Arabic Text-To-Speech System Using Transfer Learning  
**Author:** Mohamed Salem Ebnou Echvagha Oubeid (C34613)  
**Module:** NLP Dialects — Master M1 AI  
**Date:** June 2026

---

## Objective

In this notebook, we preprocess the audio data to prepare it for TTS model training.
The key steps are:

1. **Extract audio** from parquet to WAV files
2. **Resample** to a standard sample rate (22050 Hz)
3. **Trim silence** from start and end
4. **Extract features**: Mel spectrograms, MFCCs
5. **Validate** audio-text alignment

### Why Mel Spectrograms?

Modern TTS models (Tacotron, VITS, etc.) use **Mel spectrograms** as an
intermediate representation. A Mel spectrogram is a time-frequency representation
that mimics how the human ear perceives sound:

- **X-axis**: Time
- **Y-axis**: Mel-scaled frequency bands (emphasizing frequencies humans hear best)
- **Color intensity**: Energy at each time-frequency point

## 1. Setup

In [ ]:
# Install dependencies (for Colab)
# !pip install librosa soundfile matplotlib pandas pyarrow numpy tqdm

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import io
import os
from tqdm import tqdm

import librosa
import librosa.display
import soundfile as sf

from IPython.display import Audio, display

# Configuration
SAMPLE_RATE = 22050
N_FFT = 1024
HOP_LENGTH = 256
N_MELS = 80
TOP_DB = 20  # for silence trimming

plt.rcParams['figure.figsize'] = (12, 4)
sns.set_style('whitegrid')

print(f'Configuration:')
print(f'  Sample rate: {SAMPLE_RATE} Hz')
print(f'  FFT size: {N_FFT}')
print(f'  Hop length: {HOP_LENGTH}')
print(f'  Mel bands: {N_MELS}')

In [ ]:
# Load dataset
df = pd.read_parquet('../audios_dataset.parquet')
print(f'Loaded {len(df)} samples')

## 2. Audio Extraction & Resampling

We extract audio bytes from the parquet file, decode them, and resample to
22050 Hz — the standard sample rate for TTS models like Tacotron2 and VITS.

In [ ]:
def extract_audio(audio_dict, target_sr=SAMPLE_RATE):
    """Extract and resample audio from dataset dictionary."""
    audio_bytes = audio_dict.get('bytes', b'')
    if not audio_bytes:
        return None, None
    try:
        audio, sr = sf.read(io.BytesIO(audio_bytes))
        audio = audio.astype(np.float32)
        # Convert stereo to mono if needed
        if len(audio.shape) > 1:
            audio = np.mean(audio, axis=1)
        # Resample if needed
        if sr != target_sr:
            audio = librosa.resample(audio, orig_sr=sr, target_sr=target_sr)
        return audio, target_sr
    except Exception as e:
        print(f'Error: {e}')
        return None, None

# Test with first sample
audio, sr = extract_audio(df['audio'].iloc[0])
print(f'Test extraction: shape={audio.shape}, sr={sr}, duration={len(audio)/sr:.2f}s')

In [ ]:
# Extract all audio samples
audios = []
valid_indices = []
durations_before = []

for i in tqdm(range(len(df)), desc='Extracting audio'):
    audio, sr = extract_audio(df['audio'].iloc[i])
    if audio is not None and len(audio) > 0:
        audios.append(audio)
        valid_indices.append(i)
        durations_before.append(len(audio) / sr)

print(f'\nSuccessfully extracted: {len(audios)} / {len(df)} samples')
print(f'Total duration: {sum(durations_before):.1f}s ({sum(durations_before)/60:.1f} min)')

## 3. Silence Trimming

Audio recordings often have silence at the beginning and end.
Trimming silence helps the TTS model focus on actual speech content.

In [ ]:
# Trim silence from all samples
trimmed_audios = []
durations_after = []

for audio in tqdm(audios, desc='Trimming silence'):
    trimmed, _ = librosa.effects.trim(audio, top_db=TOP_DB)
    trimmed_audios.append(trimmed)
    durations_after.append(len(trimmed) / SAMPLE_RATE)

time_saved = sum(durations_before) - sum(durations_after)
print(f'\nBefore trimming: {sum(durations_before):.1f}s total')
print(f'After trimming:  {sum(durations_after):.1f}s total')
print(f'Time removed:    {time_saved:.1f}s ({100*time_saved/sum(durations_before):.1f}%)')

In [ ]:
# Visualize before/after trimming for a sample
sample_idx = 0
fig, axes = plt.subplots(2, 1, figsize=(14, 6))

t_before = np.linspace(0, len(audios[sample_idx])/SAMPLE_RATE, len(audios[sample_idx]))
axes[0].plot(t_before, audios[sample_idx], color='#1565C0', linewidth=0.5)
axes[0].set_title(f'Before Trimming — Duration: {durations_before[sample_idx]:.2f}s')
axes[0].set_ylabel('Amplitude')

t_after = np.linspace(0, len(trimmed_audios[sample_idx])/SAMPLE_RATE, len(trimmed_audios[sample_idx]))
axes[1].plot(t_after, trimmed_audios[sample_idx], color='#2E7D32', linewidth=0.5)
axes[1].set_title(f'After Trimming — Duration: {durations_after[sample_idx]:.2f}s')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Amplitude')

plt.tight_layout()
plt.savefig('../results/figures/silence_trimming_example.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Feature Extraction: Mel Spectrograms

### How Mel Spectrograms Work

1. Audio waveform is split into overlapping frames (windows)
2. FFT (Fast Fourier Transform) is applied to each frame
3. Frequency bins are grouped into Mel-scale filter banks
4. Log compression is applied (dB scale)

The result is a 2D matrix: **[n_mels × time_frames]**

This is the standard input format for modern TTS models.

In [ ]:
def compute_mel_spectrogram(audio, sr=SAMPLE_RATE):
    """Compute log-Mel spectrogram from audio waveform."""
    mel = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_fft=N_FFT,
        hop_length=HOP_LENGTH, n_mels=N_MELS
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    return mel_db

def compute_mfcc(audio, sr=SAMPLE_RATE, n_mfcc=13):
    """Compute MFCCs (Mel-Frequency Cepstral Coefficients)."""
    return librosa.feature.mfcc(
        y=audio, sr=sr, n_mfcc=n_mfcc,
        n_fft=N_FFT, hop_length=HOP_LENGTH
    )

# Compute for first sample
mel = compute_mel_spectrogram(trimmed_audios[0])
mfcc = compute_mfcc(trimmed_audios[0])
print(f'Mel spectrogram shape: {mel.shape}  (n_mels × time_frames)')
print(f'MFCC shape: {mfcc.shape}  (n_mfcc × time_frames)')

In [ ]:
# Visualize Mel spectrogram, MFCC, and waveform for multiple samples
fig, axes = plt.subplots(3, 3, figsize=(18, 12))

for col in range(3):
    audio = trimmed_audios[col]
    text = df['transcription'].iloc[valid_indices[col]]
    mel = compute_mel_spectrogram(audio)
    mfcc = compute_mfcc(audio)
    
    # Waveform
    t = np.linspace(0, len(audio)/SAMPLE_RATE, len(audio))
    axes[0, col].plot(t, audio, color='#1565C0', linewidth=0.5)
    axes[0, col].set_title(f'Waveform — "{text[:30]}..."', fontsize=9)
    axes[0, col].set_ylabel('Amplitude')
    
    # Mel spectrogram
    librosa.display.specshow(mel, y_axis='mel', x_axis='time',
                             sr=SAMPLE_RATE, hop_length=HOP_LENGTH, ax=axes[1, col])
    axes[1, col].set_title('Mel Spectrogram', fontsize=9)
    
    # MFCC
    librosa.display.specshow(mfcc, x_axis='time', sr=SAMPLE_RATE,
                             hop_length=HOP_LENGTH, ax=axes[2, col])
    axes[2, col].set_title('MFCC', fontsize=9)
    axes[2, col].set_ylabel('Coefficient')

plt.tight_layout()
plt.savefig('../results/spectrograms/feature_extraction_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to results/spectrograms/feature_extraction_samples.png')

## 5. Batch Feature Extraction

In [ ]:
# Compute Mel spectrograms for all samples
mel_specs = []
mel_lengths = []

for audio in tqdm(trimmed_audios, desc='Computing Mel spectrograms'):
    mel = compute_mel_spectrogram(audio)
    mel_specs.append(mel)
    mel_lengths.append(mel.shape[1])

print(f'\nComputed {len(mel_specs)} Mel spectrograms')
print(f'Mean spectrogram length: {np.mean(mel_lengths):.0f} frames')
print(f'Min: {min(mel_lengths)} frames, Max: {max(mel_lengths)} frames')

In [ ]:
# Visualize spectrogram length distribution
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(mel_lengths, bins=30, color='#FF5722', edgecolor='white', alpha=0.8)
ax.set_xlabel('Spectrogram Length (frames)')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Mel Spectrogram Lengths')
ax.axvline(np.mean(mel_lengths), color='red', linestyle='--',
           label=f'Mean: {np.mean(mel_lengths):.0f} frames')
ax.legend()

plt.tight_layout()
plt.savefig('../results/figures/spectrogram_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Save Processed Data

In [ ]:
# Save processed audio files as WAV
os.makedirs('../data/processed', exist_ok=True)

saved_count = 0
for i, (audio, idx) in enumerate(zip(trimmed_audios, valid_indices)):
    wav_path = f'../data/processed/hassaniya_{idx:04d}.wav'
    sf.write(wav_path, audio, SAMPLE_RATE)
    saved_count += 1

print(f'Saved {saved_count} processed WAV files to data/processed/')

In [ ]:
# Save Mel spectrograms as numpy arrays
os.makedirs('../data/processed/mels', exist_ok=True)

for i, (mel, idx) in enumerate(zip(mel_specs, valid_indices)):
    np.save(f'../data/processed/mels/hassaniya_{idx:04d}.npy', mel)

print(f'Saved {len(mel_specs)} Mel spectrogram files to data/processed/mels/')

In [ ]:
# Listen to processed samples
for i in range(min(3, len(trimmed_audios))):
    idx = valid_indices[i]
    text = df['transcription'].iloc[idx]
    print(f'\nSample {idx}: "{text}"')
    print(f'Duration: {durations_after[i]:.2f}s')
    display(Audio(trimmed_audios[i], rate=SAMPLE_RATE))

## 7. Preprocessing Summary

In [ ]:
print('=' * 60)
print('     PREPROCESSING SUMMARY')
print('=' * 60)
print(f'  Total samples processed:     {len(trimmed_audios)}')
print(f'  Sample rate:                 {SAMPLE_RATE} Hz')
print(f'  Mel spectrogram bands:       {N_MELS}')
print(f'  FFT window size:             {N_FFT}')
print(f'  Hop length:                  {HOP_LENGTH}')
print(f'  Total duration (trimmed):    {sum(durations_after):.1f}s')
print(f'  Mean duration:               {np.mean(durations_after):.2f}s')
print(f'  Mean spectrogram length:     {np.mean(mel_lengths):.0f} frames')
print(f'  Silence removed:             {time_saved:.1f}s')
print('=' * 60)
print()
print('Output files:')
print('  - data/processed/*.wav         (trimmed audio)')
print('  - data/processed/mels/*.npy    (Mel spectrograms)')
print()
print('Ready for TTS demo (Notebook 04)')

## Conclusion

We successfully preprocessed all audio samples:

1. **Extraction**: Decoded audio bytes from parquet format
2. **Resampling**: Standardized to 22050 Hz
3. **Trimming**: Removed silence from recordings
4. **Feature extraction**: Computed Mel spectrograms and MFCCs
5. **Storage**: Saved processed WAV files and Mel spectrograms

The preprocessed data is now ready for the TTS model demonstration.

**Next step**: Notebook 04 — TTS Pipeline Demo with pretrained model.